09 RAG Exam

In [45]:
%load_ext dotenv
%dotenv

In [105]:
from langchain_pdf_inspector import PdfInspectorLoader
from langchain_text_splitters import (MarkdownHeaderTextSplitter,
                                        CharacterTextSplitter)

from langchain_openai import (ChatOpenAI, 
                              OpenAIEmbeddings)

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.messages import SystemMessage
from langchain_core.prompts import (SystemMessagePromptTemplate,
                                    HumanMessagePromptTemplate, 
                                    ChatPromptTemplate)
from langchain_core.runnables import (RunnablePassthrough, 
                                      RunnableLambda,
                                        RunnableParallel,
                                      chain)
import numpy as np

In [17]:
total = PdfInspectorLoader("365_Data_Science_Courses.pdf").load()[0].metadata["total_pages"]

In [18]:
loader_pdf = PdfInspectorLoader(
    "365_Data_Science_Courses.pdf",
    mode="page",
    output="markdown",
    pages=list(range(total)),   # 0-indexed
)

In [19]:
pages_pdf = loader_pdf.load()

In [20]:
print(pages_pdf)

[Document(metadata={'source': '365_Data_Science_Courses.pdf', 'pdf_type': 'text_based', 'total_pages': 3, 'title': '365_Data_Science_Courses.pdf', 'author': 'Hristina  Hristova', 'creator': 'Microsoft® Word for Microsoft 365', 'producer': 'Microsoft® Word for Microsoft 365', 'creation_date': "D:20240703151619+03'00'", 'mod_date': "D:20240703151619+03'00'", 'page': 1, 'needs_ocr': False, 'has_tables': False, 'has_multicolumn_layout': False}, page_content='# # Introduction to Data and Data Science\n\n## ## Big Data Real-life Examples\n\nAll right! So… Where do we find big data? The answer is in increasingly more industries and companies. Here are a few notable examples.\n\nAs one of the largest online communities, ‘Facebook’ keeps track of its user’s names, personal data, photos, videos, recorded messages and so on. This means their data has a lot of variety. And with over 2 billion users worldwide, the volume of data stored on their servers is tremendous. Facebook requires real-time rep

In [21]:
pages_string = ''.join([x.page_content for x in pages_pdf])

In [28]:
pages_string = pages_string.replace('## Thanks', '### Thanks')
print(pages_string)

# # Introduction to Data and Data Science

## ## Big Data Real-life Examples

All right! So… Where do we find big data? The answer is in increasingly more industries and companies. Here are a few notable examples.

As one of the largest online communities, ‘Facebook’ keeps track of its user’s names, personal data, photos, videos, recorded messages and so on. This means their data has a lot of variety. And with over 2 billion users worldwide, the volume of data stored on their servers is tremendous. Facebook requires real-time reporting of the aggregated, anonymized voice of its users, and it applies many analytical tools for its mobile applications. This means the company is investing in boosting its real-time data processing powers, increasing the velocity of its data set.

### Great stuff!

Now, where else can we find ‘big data’?

Let’s take financial trading data for example. What happens when we record the stock price every 5 seconds? Or every single second? We get a dataset that i

In [29]:
headers_to_split_on = [
    ("#", "Course Title"),
    ("##", "Lecture Title"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers = False)

In [30]:
pages_md_split = md_splitter.split_text(pages_string)

In [32]:
len(pages_md_split)

4

In [57]:
char_splitter = CharacterTextSplitter(
  separator='.',
  chunk_size = 400,
  chunk_overlap = 0,
  keep_separator = False,
)

In [58]:
pages_char_split = char_splitter.split_documents(pages_md_split)

In [59]:
len(pages_char_split)

17

In [60]:
pages_char_split[1].page_content

'This means their data has a lot of variety. And with over 2 billion users worldwide, the volume of data stored on their servers is tremendous. Facebook requires real-time reporting of the aggregated, anonymized voice of its users, and it applies many analytical tools for its mobile applications'

In [61]:
pages_char_split[8].page_content

'They will offer them exclusive discounts to thank them for their custom and by doing so, keep them returning. On a larger scale, companies can use machine learning and past labelled data to automate the practice. And with this they can know which customers may purchase goods from them'

In [62]:
embedding = OpenAIEmbeddings(model='text-embedding-3-small')

In [63]:
vector1 = embedding.embed_query(pages_char_split[1].page_content)

In [64]:
vector8 = embedding.embed_query(pages_char_split[8].page_content)

In [65]:
np.dot(vector1, vector8)

np.float64(0.2728173639192164)

In [67]:
client = QdrantClient(path="./365-ds-courses/")

# client.delete_collection(
#     collection_name="365-ds-courses"
# )

In [68]:
client.create_collection(
    collection_name="365-ds-courses",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="365-ds-courses",
    embedding=embedding,
)

In [69]:
uuids = [str(uuid4()) for _ in range(len(pages_char_split))]

In [70]:
doc_ids = vector_store.add_documents(documents=pages_char_split, ids=uuids)

In [71]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.7})

In [74]:
retriever.invoke('Could you give me some examples of big data used in real life?')

[Document(metadata={'Course Title': '# Introduction to Data and Data Science', 'Lecture Title': '## Big Data Real-life Examples', '_id': '8cafc88d-cfa7-4802-83cb-05099ef6f8a9', '_collection_name': '365-ds-courses'}, page_content='# # Introduction to Data and Data Science  \n## ## Big Data Real-life Examples  \nAll right! So… Where do we find big data? The answer is in increasingly more industries and companies. Here are a few notable examples.  \nAs one of the largest online communities, ‘Facebook’ keeps track of its user’s names, personal data, photos, videos, recorded messages and so on'),
 Document(metadata={'Course Title': '## Create Easily Printable Excel Documents', '_id': '0e6e04a2-8653-404e-bb58-77e9edbe7170', '_collection_name': '365-ds-courses'}, page_content='Then I’ll go to the menu bar and select “Page Layout,” where we have the “Print Area” menu. I’ll click on it and select “Set Print Area.”  \nSee how Excel put a border to the print area of our sheet? If we print it, onl

In [98]:
runnable_retriever = RunnableLambda(lambda x: retriever.invoke(x))

In [92]:
message_template1 = SystemMessagePromptTemplate.from_template('''
You are a helpful Q&A chatbot at the 365 Data Science online educational platform. 
The platform offers video courses on data science topics. 
Your task is to provide students with exhaustive answers to their questions.

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture and the course this context is taken from in the format:
Resources: 
1. Lecture "Lecture Title" from the "Course Title" course
…
where "Lecture Title" and "Course Title" should be substituted with the respective titles of all resource lectures.
''')

In [93]:
message_template2 = HumanMessagePromptTemplate.from_template('''{question}''')

In [94]:
chat_template = ChatPromptTemplate.from_messages([message_template1, 
                                                  message_template2])

In [95]:
chat = ChatOpenAI(model='gpt-6-luna', seed=365, temperature=1, max_completion_tokens=1000)

In [96]:
str_parser = StrOutputParser()

In [108]:
chain = (RunnableParallel({'context': runnable_retriever, 'question': RunnablePassthrough()})
         | chat_template
         | chat
         | str_parser
)

In [111]:
response= chain.invoke('How can we print documents in Microsoft Excel?')

In [112]:
print(response)

To make an Excel document printable, organize the worksheet with printing in mind and set a **print area** so Excel knows which cells to include when printing. The course context doesn’t specify the exact menu steps for setting the print area or starting the print job.

Resources:
1. Lecture "Create Easily Printable Excel Documents" from the "Create Easily Printable Excel Documents" course
